Final wird nun auch noch der NDVI hinzugefügt und zwar immer der NDVI in den 50 metern um die sensoren, also das Mikroklima. Dazu habe ich davor mit dem GRASS-Tool in QGIS die Werte auf 50m gemittelt, um eine gewisse range um den sensor zu haben und nicht nur die 10 meter.

In [1]:
import os
import pandas as pd
import rasterio
import numpy as np

# =========================================================
# 1. Pfade definieren
# =========================================================
master_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exp2_Features_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"
ndvi_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Vorbereitete Daten für RF\50m_AOI_NDVI_S2_11082024.tif"

# Ausgabedatei - Deine endgültige Feature-Masterdatei!
output_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exp2_Features_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"

# =========================================================
# 2. Master-Datensatz laden
# =========================================================
print("Lade Masterdatei...")
df_master = pd.read_csv(master_file, sep=";", decimal=",")

# Sicherstellen, dass UTM-Koordinaten als numerische Werte vorliegen
for col in ["x_utm", "y_utm"]:
    if df_master[col].dtype == object:
        df_master[col] = df_master[col].str.replace(",", ".", regex=False)
    df_master[col] = pd.to_numeric(df_master[col], errors="coerce")

# Neue Spalte initialisieren
df_master["ndvi"] = np.nan

# =========================================================
# 3. NDVI-Daten extrahieren
# =========================================================
if not os.path.exists(ndvi_path):
    print(f"WARNUNG: NDVI-Datei nicht gefunden -> {ndvi_path}")
else:
    print(f"Extrahiere NDVI-Werte aus {os.path.basename(ndvi_path)}...")

    # Filtere nur Zeilen, die gültige Koordinaten haben
    valid_mask = df_master["x_utm"].notna() & df_master["y_utm"].notna()

    # Erstelle eine Liste von (x, y) Koordinaten-Tupeln
    coords_list = list(zip(df_master.loc[valid_mask, "x_utm"], df_master.loc[valid_mask, "y_utm"]))

    if len(coords_list) > 0:
        with rasterio.open(ndvi_path) as src:
            nodata_val = src.nodata
            sampled_values = []

            # Werte anstechen
            for val in src.sample(coords_list):
                v = val[0]
                # NoData und unnatürliche Extremwerte rausfiltern
                # (NDVI liegt normal zwischen -1 und 1, manchmal in skalierten TIFs auch bis +/- 10000)
                if v == nodata_val or v > 1e6 or v < -1e6:
                    sampled_values.append(np.nan)
                else:
                    sampled_values.append(v)

            # Werte exakt den gültigen Zeilen zuweisen
            df_master.loc[valid_mask, "ndvi"] = sampled_values

# =========================================================
# 4. Speichern
# =========================================================
df_master.to_csv(output_file, index=False, sep=";", decimal=",")

print("-" * 60)
print(f"ERFOLG! Dein Datensatz ist nun absolut komplett (inkl. NDVI) und liegt hier:\n{output_file}")
print("-" * 60)

Lade Masterdatei...
Extrahiere NDVI-Werte aus 50m_AOI_NDVI_S2_11082024.tif...
------------------------------------------------------------
ERFOLG! Dein Datensatz ist nun absolut komplett (inkl. NDVI) und liegt hier:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exp2_Features_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv
------------------------------------------------------------
